# colab_08b — Remediation: intersection-only HVGs

## Motivation

`colab_09` 1d gave a decisive verdict on the colab_08 red flag: cluster 0 (23,737 cells, 98.7% organoid-pure) **is cortical RG**. The 298 fetal cells inside it were 75% RG by `cell_type_coarse`, an enrichment of **6.66× over the 11.2% Bhaduri 2021 RG baseline**.

Arithmetic:
- 11.2% × 100k = ca. 11,200 fetal RG total in the 2021 subsample.
- Only ca. 223 fetal RG (= 0.298k × 0.748) ended up in cluster 0.
- The other **ca. 10,977 fetal RG are elsewhere** on the UMAP — Harmony placed them in different clusters than the organoid RG, even though they're the same cell type.

This is under-correction: same cell type, two clusters.

## Cheapest fix: intersection-only HVGs

`colab_08` used `sc.pp.highly_variable_genes(..., n_top_genes=2000, batch_key='dataset')`, which ranks HVGs per batch then combines — pulling in 1,249 genes that are HV in only one dataset. Some of those one-dataset-only HVGs are likely organoid-specific drivers (stress, choroid plexus, protocol artifacts) or fetal-specific drivers (microglia, OPCs, vascular signatures) that pull each population into its own neighborhood, leaving Harmony with too little shared signal to mix them.

**This notebook restricts the HVG set to genes that are HV in BOTH datasets** (`highly_variable_nbatches == 2`) — the 751-gene subset from colab_08's run. Forces Harmony to align on shared variable genes only. Cheapest of three escalating remediation paths:

1. **Intersection-only HVGs** ← this notebook
2. Increase Harmony `theta` (default 2 → 4–6) — only if (1) doesn't fix it
3. Switch to scVI — only if (1) and (2) both fail

## Outputs

- `integrated_100k_harmony_intersectHVG.h5ad` on Drive (separate from colab_08's output, both kept for comparison).
- Diagnostic at the end re-runs colab_09's 1d check on the new clustering: which cluster contains the most organoid RG, and what fraction of its fetal contingent is RG-labeled.

## 0. Setup

### 0a — Install dependencies, mount Drive, import packages

Installs `scanpy`, `leidenalg`, `harmonypy` (none are pre-installed on standard Colab images). Mounts Google Drive (project lives at `MyDrive/brain-organoid-trajectories`), imports the scanpy stack, and defines `PATHS` for both inputs and the integration output.

In [ ]:
!pip install -q scanpy leidenalg harmonypy

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import harmonypy as hm
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(6, 5))

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'bhaduri_2020_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'),
    'bhaduri_2021_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad'),
    'integrated_out':    os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_100k_harmony_intersectHVG.h5ad'),
}
os.makedirs(os.path.dirname(PATHS['integrated_out']), exist_ok=True)
print('scanpy', sc.__version__, '| anndata', ad.__version__, '| harmonypy', hm.__version__)

## 1. Load both 100k subsamples

### 1a — Read both h5ads from Drive

Loads each AnnData and prints shape, raw status, and obs columns. The two datasets store counts differently — we unify in section 2.

In [ ]:
adata_2020 = sc.read_h5ad(PATHS['bhaduri_2020_100k'])
adata_2021 = sc.read_h5ad(PATHS['bhaduri_2021_100k'])

print(f'Bhaduri 2020: {adata_2020.shape[0]:,} cells x {adata_2020.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2020.raw is not None else "None"})')
print(f'Bhaduri 2021: {adata_2021.shape[0]:,} cells x {adata_2021.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2021.raw is not None else "None"})')
print()
print('Bhaduri 2020 obs cols:', list(adata_2020.obs.columns))
print('Bhaduri 2021 obs cols:', list(adata_2021.obs.columns))

## 2. Bring both `.X` matrices to the same processing stage (log-normalized)

The two subsamples store data asymmetrically:
- **Bhaduri 2020**: `.raw` is a standard scanpy snapshot taken *after* normalize_total + log1p in colab_01. So `.raw.X` is log(CP10K+1) data — **not raw counts** (this contradicts an earlier session note; verified in 2b on 2026-04-25).
- **Bhaduri 2021**: `raw=False` from colab_07 — `.X` holds genuine raw integer counts.

Goal: bring both to log(CP10K+1) on the full gene space before concat. 2020 only needs `.X = .raw.X`; 2021 needs normalize_total + log1p applied. Old colab_03's blanket `adata.X = adata.raw.X` would crash on 2021.

### 2a — For Bhaduri 2020: copy `.raw.X` (log-normalized full-gene matrix) into `.X`

Replaces 2020's scaled-HVG `.X` (from colab_02) with the log-normalized full-gene snapshot from `.raw`. After this cell 2020 is in log(CP10K+1) on 16,774 genes.

In [ ]:
# 2020: replace .X with raw counts from .raw.X
if adata_2020.raw is None:
    raise ValueError('Bhaduri 2020 .raw is unexpectedly None — re-check colab_07 output')
adata_2020.X = adata_2020.raw.X.copy()

# 2021: .X is already counts (raw=False from colab_07). Warn if .raw was set.
if adata_2021.raw is not None:
    print('WARN: Bhaduri 2021 .raw unexpectedly set — still using .X (assumed counts)')

# ensure both are sparse CSR
for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    if not sp.issparse(a.X):
        a.X = sp.csr_matrix(a.X)
    elif a.X.format != 'csr':
        a.X = a.X.tocsr()
    print(f'Bhaduri {name}: dtype={a.X.dtype}, format={a.X.format}, nnz={a.X.nnz:,}')

### 2b — Inspect `.X` ranges to confirm processing stage

Sanity check: 2020's `.X` should now look like log-normalized data (float, max ca. 6–8, non-integer). 2021's `.X` should still be raw integer counts (max in tens or hundreds, integer-like).

In [ ]:
for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    n = min(1000, a.X.nnz)
    sample = a.X.data[:n]
    is_int_like = np.allclose(sample, sample.astype(int))
    print(f'Bhaduri {name}: min={sample.min()}, max={sample.max()}, '
          f'dtype={sample.dtype}, integer-like={is_int_like}')

### 2c — Normalize + log-transform Bhaduri 2021 to match 2020

`sc.pp.normalize_total(target_sum=1e4)` + `sc.pp.log1p`. Brings 2021 from raw integer counts into log(CP10K+1) on the full gene space, the same stage 2020 is at after 2a.

In [ ]:
sc.pp.normalize_total(adata_2021, target_sum=1e4)
sc.pp.log1p(adata_2021)
print(f'2021 .X after norm+log: dtype={adata_2021.X.dtype}, '
      f'min={adata_2021.X.min():.3f}, max={adata_2021.X.max():.2f}')

## 3. Shared gene space

### 3a — Intersect `var_names` across datasets

Integration requires identical gene panels. Expected from Session 20 sanity check: ca. 16,768 shared genes — the 6 Bhaduri-2020-only genes are `.1`-suffixed Cell Ranger duplicates and biologically irrelevant. Subset both datasets to the shared genes in sorted order.

In [ ]:
shared_genes = adata_2020.var_names.intersection(adata_2021.var_names).sort_values()

print(f'Bhaduri 2020 genes: {adata_2020.shape[1]:,}')
print(f'Bhaduri 2021 genes: {adata_2021.shape[1]:,}')
print(f'Shared:             {len(shared_genes):,}')
print(f'Lost from 2020:     {adata_2020.shape[1] - len(shared_genes):,}')
print(f'Lost from 2021:     {adata_2021.shape[1] - len(shared_genes):,}')

adata_2020 = adata_2020[:, shared_genes].copy()
adata_2021 = adata_2021[:, shared_genes].copy()

print()
print(f'After subset — 2020: {adata_2020.shape}')
print(f'After subset — 2021: {adata_2021.shape}')

## 4. Concatenate with dataset label

### 4a — Prefix barcodes and concat

Prefix obs names with `b2020_` / `b2021_` so combined barcodes are globally unique. `ad.concat(..., label='dataset')` adds an obs column named `dataset` with values `bhaduri_2020` / `bhaduri_2021` — this becomes Harmony's batch key.

Dataset-specific obs columns (e.g. `protocol`, `age_week`, `cell_type_coarse`, `age_gw`, `donor`, `area_ucsc`) are not in both objects, so they will be `NaN`-filled or dropped depending on `merge` strategy. We re-attach what we need post-integration in colab_09.

In [ ]:
adata_2020.obs_names = 'b2020_' + adata_2020.obs_names
adata_2021.obs_names = 'b2021_' + adata_2021.obs_names
print('First 3 barcodes 2020:', adata_2020.obs_names[:3].tolist())
print('First 3 barcodes 2021:', adata_2021.obs_names[:3].tolist())

adata = ad.concat(
    {'bhaduri_2020': adata_2020, 'bhaduri_2021': adata_2021},
    axis=0,
    join='outer',
    label='dataset',
    merge='same',
    index_unique=None,
)
adata.obs['dataset'] = adata.obs['dataset'].astype('category')

# free per-dataset copies
del adata_2020, adata_2021

print()
print(f'Combined: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')
print(adata.obs['dataset'].value_counts())

## 5. Select highly variable genes

Both datasets are already log(CP10K+1) (after 2a + 2c), so no normalization needed here. Just HVG selection per batch.

### 5a — Confirm combined `.X` is log-normalized

Quick sanity check before HVG selection. Combined `.X` should be float, max ca. 6–8.

In [ ]:
print(f'Combined .X: dtype={adata.X.dtype}, '
      f'min={adata.X.min():.3f}, max={adata.X.max():.2f}, '
      f'mean={adata.X.mean():.3f}')

### 5b — Select intersection HVGs (highly variable in BOTH datasets)

Run `sc.pp.highly_variable_genes` with `batch_key='dataset'` exactly as in colab_08, then filter `highly_variable` to genes with `highly_variable_nbatches == 2` (HV in both Bhaduri 2020 and Bhaduri 2021). Expected output: ca. 751 genes (the intersection count from colab_08's run).

`flavor='seurat'` works on log-normalized `.X`. Save `.raw = adata` (full 16,768-gene matrix) before subsetting `.X` so marker plots later can `use_raw=True`.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor='seurat',
    batch_key='dataset',
)

# Restrict to intersection: HV in both batches
adata.var['highly_variable_intersect'] = (
    adata.var['highly_variable'] & (adata.var['highly_variable_nbatches'] == 2)
)
n_union     = adata.var['highly_variable'].sum()
n_intersect = adata.var['highly_variable_intersect'].sum()
print(f'Union HVGs (colab_08 setting):       {n_union:,}')
print(f'Intersection HVGs (this notebook):   {n_intersect:,}')

adata.raw = adata
adata = adata[:, adata.var['highly_variable_intersect']].copy()
print(f'After subset, .X shape: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')

## 6. Scale and PCA

### 6a — Scale (clip max=10) and run 30-component PCA

Matches Session 11's choice of 30 PCs for the unbalanced run. Note: `sc.pp.scale` densifies `.X` (HVG × cells), expected ca. 1.5 GB at 200k × 2000 × 4 bytes.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=30, svd_solver='arpack', random_state=42)
print(f'X_pca shape: {adata.obsm["X_pca"].shape}')
print(f'Variance ratio (PC1–5): {np.round(adata.uns["pca"]["variance_ratio"][:5], 4)}')
print(f'Cumulative variance through PC30: {adata.uns["pca"]["variance_ratio"].sum():.3f}')

## 7. Harmony batch correction

### 7a — Run Harmony directly (workaround for sc.external bug)

`sc.external.pp.harmony_integrate` has a shape bug in current scanpy/harmonypy versions (Session 11). We call `hm.run_harmony` directly. The memory's note from colab_03 was "assign `ho.Z_corr` without transpose" — we add a shape assertion to catch any version-driven flip.

In [ ]:
ho = hm.run_harmony(
    adata.obsm['X_pca'],
    meta_data=adata.obs,
    vars_use=['dataset'],
    max_iter_harmony=50,
    random_state=42,
)
Z = ho.Z_corr
if Z.shape != adata.obsm['X_pca'].shape:
    print(f'Note: Z_corr shape {Z.shape} != X_pca shape {adata.obsm["X_pca"].shape} — transposing')
    Z = Z.T
assert Z.shape == adata.obsm['X_pca'].shape, f'Harmony output shape mismatch: {Z.shape}'
adata.obsm['X_pca_harmony'] = Z
print(f'X_pca_harmony shape: {adata.obsm["X_pca_harmony"].shape}')

### 7b — Diagnostic: did Harmony actually shift anything?

When Harmony converges in very few iterations (e.g., 2 instead of the typical 20–40), there are two possible explanations: (1) inputs were already well-aligned and Harmony correctly applied small corrections to finish, or (2) Harmony hit the relative-change threshold (`epsilon_harmony=0.0001`) prematurely and under-corrected. This cell measures the per-cell shift between `X_pca` and `X_pca_harmony` to distinguish the two.

Interpretation rules of thumb:
- **Mean abs shift < 0.01**: Harmony barely moved anything → likely under-correction, suspect.
- **Mean abs shift in 0.1–2.0 range**: meaningful per-cell adjustment, real work done.
- **Std of shifts ca. mean shift**: heterogeneous corrections (different cells moved by different amounts) — what real integration looks like. A constant std near zero would mean a rigid translation, not real integration.
- **Max shift comparable to typical PC magnitude**: a small subset of cells (most batch-discriminative pre-Harmony) moved a lot, expected.

This is a sanity check before trusting the embedding downstream — 9a (UMAP by dataset) is still the visual verdict.

In [ ]:
diff = adata.obsm['X_pca_harmony'] - adata.obsm['X_pca']
print(f'Mean abs shift per cell: {np.abs(diff).mean():.4f}')
print(f'Max shift any cell:      {np.abs(diff).max():.4f}')
print(f'Std of shifts:           {diff.std():.4f}')

## 8. Neighbor graph, UMAP, Leiden

### 8a — Compute neighbor graph on Harmony embedding

`use_rep='X_pca_harmony'` so neighbors reflect batch-corrected distances. `n_neighbors=15` is scanpy default.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca_harmony', random_state=42)
print('Neighbor graph computed on X_pca_harmony.')

### 8b — UMAP

2D embedding for inspection. Random state fixed for reproducibility.

In [ ]:
sc.tl.umap(adata, random_state=42)
print(f'X_umap shape: {adata.obsm["X_umap"].shape}')

### 8c — Leiden clustering at resolution 0.5

Matches Session 11. Expected count: ca. 15–25 clusters.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, random_state=42)
n_clusters = adata.obs['leiden'].nunique()
print(f'Leiden clusters: {n_clusters}')
print(adata.obs['leiden'].value_counts().sort_index())

## 9. Visual diagnostics

### 9a — UMAP colored by `dataset` (mixing check)

If Harmony worked, the two colors should intermix within each cell-type region. Stripes / dataset-segregated regions = Harmony failed or under-corrected.

In [ ]:
sc.pl.umap(adata, color='dataset', frameon=False, save='_dataset.png', show=True)

### 9b — UMAP colored by Leiden cluster

Cluster geography. We assign cell types in colab_09 (next notebook).

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False, save='_leiden.png', show=True)

### 9c — Lineage marker genes

Sanity-check that markers land in expected regions:
- **SOX2**, **PAX6** — radial glia
- **EOMES** — intermediate progenitors
- **TBR1**, **NEUROD2** — excitatory neurons
- **GAD1**, **GAD2** — GABAergic interneurons
- **GFAP** — astrocytes
- **MKI67** — cycling cells

Scattered/diffuse markers → over-integration. `use_raw=True` reads from the full 16,768-gene `.raw` matrix, not the 2,000-HVG `.X`.

**Two outputs per marker run:**
1. A combined 3×3 grid panel (`_markers.png`) — good for at-a-glance comparison.
2. Individual full-size plots per gene (`_marker_<GENE>.png`) — needed for accurate per-gene spatial reads. The grid panels compress to thumbnails when reviewed; individual plots preserve resolution and make it possible to verify exactly where each marker localizes.

In [ ]:
markers = ['SOX2', 'PAX6', 'EOMES', 'TBR1', 'NEUROD2', 'GAD1', 'GAD2', 'GFAP', 'MKI67']
present = [g for g in markers if g in adata.raw.var_names]
missing = [g for g in markers if g not in adata.raw.var_names]
if missing:
    print(f'Markers missing from .raw: {missing}')

# combined grid — good for at-a-glance comparison
sc.pl.umap(adata, color=present, use_raw=True, ncols=3, frameon=False,
           save='_markers.png', show=True)

# individual full-size plots — needed for accurate per-gene interpretation
for gene in present:
    sc.pl.umap(adata, color=gene, use_raw=True, frameon=False, size=8,
               save=f'_marker_{gene}.png', show=True)

### 9d — Cluster × dataset composition

Percentage of each dataset within each Leiden cluster. Values near 50/50 mean Harmony mixed that cluster well; clusters > 95% one dataset are candidates for batch artifacts — unlike the colab_03 Zhong run, both Bhaduri datasets should largely overlap, so heavily skewed clusters here would be more concerning than in colab_03.

In [ ]:
comp = pd.crosstab(adata.obs['leiden'], adata.obs['dataset'], normalize='index') * 100
comp = comp.round(1)
comp['total_cells'] = adata.obs['leiden'].value_counts().sort_index().values
print(comp)

## 10. Save integrated object

### 10a — Write `integrated_100k_harmony_intersectHVG.h5ad` to Drive

Stored alongside `integrated_100k_harmony.h5ad` (colab_08's output) so we can directly compare clusterings. Predicted size: ca. 4–5 GB (smaller HVG count → smaller `.X`).

In [ ]:
adata.write_h5ad(PATHS['integrated_out'])
size_gb = os.path.getsize(PATHS['integrated_out']) / 1e9
print(f'Saved: {PATHS["integrated_out"]}')
print(f'Size:  {size_gb:.2f} GB')
print(f'Final shape: {adata.shape}')

## 11. Cluster-0 RG remediation check

### 11a — Find the dominant RG cluster

Identify which cluster has the largest organoid PAX6⁺ SOX2⁺ MKI67⁺ RG signature in the new clustering. We look at the cluster with the most Bhaduri 2020 cells in the cortical-RG marker zone — proxied here by mean PAX6 + SOX2 + MKI67 expression per cluster, taken on `.raw` (full-gene log-normalized).

In [ ]:
rg_markers_check = ['PAX6', 'SOX2', 'MKI67', 'VIM', 'NES', 'FABP7']
rg_idx = [adata.raw.var_names.get_loc(g) for g in rg_markers_check if g in adata.raw.var_names]
rg_score_per_cell = adata.raw.X[:, rg_idx]
if hasattr(rg_score_per_cell, 'toarray'):
    rg_score_per_cell = rg_score_per_cell.toarray()
rg_score_per_cell = rg_score_per_cell.mean(axis=1)

import pandas as pd
df = pd.DataFrame({
    'leiden':     adata.obs['leiden'].values,
    'rg_score':   rg_score_per_cell,
    'dataset':    adata.obs['dataset'].values,
})
cluster_rg = df.groupby('leiden')['rg_score'].mean().sort_values(ascending=False)
print('Top 5 clusters by mean RG-marker score:')
print(cluster_rg.head().round(3).to_string())

rg_cluster = cluster_rg.index[0]
print(f'\nDominant RG cluster: {rg_cluster}')
print(df[df['leiden']==rg_cluster]['dataset'].value_counts().to_string())

### 11b — Fetal cell_type_coarse enrichment in the dominant RG cluster

Same diagnostic as colab_09 1d, applied to the new dominant RG cluster. **Pass criterion** for intersection-HVG remediation:
- Fetal cell count in dominant RG cluster substantially higher than colab_08's 298 (target: ≳ a few thousand, since ca. 11,200 fetal RG should distribute toward this cluster if mixing works).
- Fraction of all 2020 cells in this cluster ≈ fraction of all 2021 RG in this cluster (similar absolute mixing).
- RG enrichment ratio in the fetal contingent stays high (≳ 5×) — confirms it's still an RG cluster, not a degenerated mixed cluster.

If both fetal count and RG enrichment hold up, intersection HVGs fixed it. If fetal count is still ca. 300, escalate to Harmony `theta`.

In [ ]:
mask_rg     = (adata.obs['leiden'] == rg_cluster).values
mask_2021   = (adata.obs['dataset'] == 'bhaduri_2021').values
mask_2020   = (adata.obs['dataset'] == 'bhaduri_2020').values
mask_in_2021 = mask_rg & mask_2021
mask_in_2020 = mask_rg & mask_2020

print(f'Cluster {rg_cluster} composition:')
print(f'  bhaduri_2020: {mask_in_2020.sum():,} cells '
      f'(= {mask_in_2020.sum() / mask_2020.sum() * 100:.1f}% of all 2020)')
print(f'  bhaduri_2021: {mask_in_2021.sum():,} cells '
      f'(= {mask_in_2021.sum() / mask_2021.sum() * 100:.1f}% of all 2021)')
print(f'  colab_08 cluster 0 had: 23,439 / 298   (23.4% of 2020 / 0.3% of 2021)')
print()

baseline   = adata.obs.loc[mask_2021,    'cell_type_coarse'].value_counts(normalize=True)
in_cluster = adata.obs.loc[mask_in_2021, 'cell_type_coarse'].value_counts(normalize=True)
enrichment = (in_cluster / baseline).sort_values(ascending=False)

print(f'Bhaduri 2021 baseline cell_type_coarse (n = {mask_2021.sum():,}):')
print(baseline.round(3).to_string())
print()
print(f'Cluster {rg_cluster} — Bhaduri 2021 cells only (n = {mask_in_2021.sum():,}):')
print(in_cluster.round(3).to_string())
print()
print(f'Enrichment ratio (>1 = enriched):')
print(enrichment.round(2).to_string())

### 11c — Verdict and routing

Decision rule:
- **Pass** (fetal RG count ≳ several thousand AND RG enrichment ≳ 5× AND ca. 41% pure-cluster fraction drops substantially) → intersection HVGs fixed it. Proceed to colab_10 (full annotation) on this h5ad.
- **Partial** (fetal count up but still skewed, or pure-cluster fraction only modestly improved) → escalate to Harmony `theta=4` (or higher). Add as section 12 here, or fork colab_08c.
- **Fail** (fetal count still ca. 300, dominant RG cluster still 98%+ organoid-pure) → switch to scVI. Fork colab_08d using the scvi-tools workflow.

In [ ]:
# Summary print
n_pure_2020 = sum(1 for cl in adata.obs['leiden'].unique()
                  if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2020').mean() > 0.95)
n_pure_2021 = sum(1 for cl in adata.obs['leiden'].unique()
                  if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2021').mean() > 0.95)
total_pure_cells = sum(
    (adata.obs['leiden']==cl).sum()
    for cl in adata.obs['leiden'].unique()
    if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2020').mean() > 0.95
    or (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2021').mean() > 0.95
)

print('=' * 70)
print('REMEDIATION SUMMARY — intersection HVGs')
print('=' * 70)
print(f'HVG count:                        {adata.shape[1]:,}  (vs colab_08\'s 2,000)')
print(f'Leiden clusters:                  {adata.obs["leiden"].nunique()}  (vs colab_08\'s 21)')
print(f'>95%-pure clusters: 2020={n_pure_2020}, 2021={n_pure_2021}')
print(f'Cells in >95%-pure clusters:      {total_pure_cells:,} '
      f'({total_pure_cells/len(adata)*100:.0f}%)  (vs colab_08\'s ca. 41%)')
print(f'Dominant RG cluster:              {rg_cluster}')
print(f'  fetal cells in RG cluster:      {mask_in_2021.sum():,}  (vs colab_08\'s 298)')
print(f'  fetal RG enrichment:            {enrichment.get("RG", 0):.2f}x  (vs colab_08\'s 6.66x)')
print('=' * 70)